# Build a Voice Assistant with Claude and MiniMax TTS

This cookbook demonstrates how to combine **Claude** (Anthropic) for intelligent text generation with **MiniMax TTS** for high-quality speech synthesis to build a voice assistant.

[MiniMax](https://www.minimax.io/) provides state-of-the-art text-to-speech (TTS) and large language model APIs. Their TTS API (`speech-2.8-hd`) produces natural-sounding speech with support for multiple voices and streaming output.

## What You'll Learn

- How to use Claude to generate conversational responses
- How to call the MiniMax TTS API to synthesize speech
- How to parse MiniMax's SSE (Server-Sent Events) streaming response
- How to save and play back audio
- How to build a multi-turn voice conversation pipeline

## Prerequisites

You'll need:
- An **Anthropic API key** from [console.anthropic.com](https://console.anthropic.com/settings/keys)
- A **MiniMax API key** from [platform.minimax.io](https://platform.minimax.io/)

## Step 1: Install Dependencies

In [ ]:
!pip install anthropic requests python-dotenv

## Step 2: Configure API Keys

Set your API keys below or load them from a `.env` file.

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()

ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "your_anthropic_api_key_here")
MINIMAX_API_KEY = os.environ.get("MINIMAX_API_KEY", "your_minimax_api_key_here")

assert ANTHROPIC_API_KEY != "your_anthropic_api_key_here", "Please set your ANTHROPIC_API_KEY"
assert MINIMAX_API_KEY != "your_minimax_api_key_here", "Please set your MINIMAX_API_KEY"

print("API keys loaded successfully!")

## Step 3: Generate Text with Claude

First, let's use Claude to generate a conversational response.

In [ ]:
import anthropic

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)


def get_claude_response(user_message: str, conversation_history: list | None = None) -> str:
    """Get a response from Claude for the given user message."""
    messages = conversation_history or []
    messages = messages + [{"role": "user", "content": user_message}]

    response = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=300,
        system=(
            "You are a friendly and helpful voice assistant. "
            "Keep your responses concise (2-3 sentences) since they will be "
            "converted to speech. Avoid using markdown, bullet points, or "
            "special characters."
        ),
        messages=messages,
    )

    return response.content[0].text


# Test Claude response
test_response = get_claude_response("What is the capital of France?")
print(f"Claude: {test_response}")

## Step 4: Convert Text to Speech with MiniMax TTS

Now let's use the MiniMax TTS API to synthesize the Claude response into audio.

MiniMax's TTS API streams audio as Server-Sent Events (SSE). Each event contains a hex-encoded audio chunk that we decode and assemble into the final audio file.

### Available Voices

| Voice ID | Gender | Style |
|----------|--------|-------|
| `English_Graceful_Lady` | Female | Graceful, warm |
| `English_Insightful_Speaker` | Male | Calm, authoritative |
| `English_radiant_girl` | Female | Energetic, bright |
| `English_Persuasive_Man` | Male | Confident, persuasive |
| `English_Lucky_Robot` | Neutral | Sci-fi, robotic |
| `English_expressive_narrator` | Male | Expressive, storytelling |

In [ ]:
import json

import requests

MINIMAX_TTS_URL = "https://api.minimax.io/v1/t2a_v2"


def text_to_speech(
    text: str,
    voice_id: str = "English_Graceful_Lady",
    model: str = "speech-2.8-hd",
    output_path: str = "output.mp3",
) -> str:
    """
    Convert text to speech using MiniMax TTS API.

    Args:
        text: The text to synthesize.
        voice_id: MiniMax voice ID to use.
        model: TTS model (speech-2.8-hd or speech-2.8-turbo).
        output_path: Path to save the output audio file.

    Returns:
        Path to the saved audio file.
    """
    headers = {
        "Authorization": f"Bearer {MINIMAX_API_KEY}",
        "Content-Type": "application/json",
    }

    payload = {
        "model": model,
        "text": text,
        "stream": True,
        "voice_setting": {
            "voice_id": voice_id,
            "speed": 1.0,
            "vol": 1.0,
            "pitch": 0,
        },
        "audio_setting": {
            "sample_rate": 32000,
            "bitrate": 128000,
            "format": "mp3",
            "channel": 1,
        },
    }

    response = requests.post(
        MINIMAX_TTS_URL, headers=headers, json=payload, stream=True, timeout=120
    )
    response.raise_for_status()

    # Parse SSE stream and collect hex-encoded audio chunks
    audio_chunks = []
    buffer = ""

    for raw_bytes in response.iter_content(chunk_size=None):
        buffer += raw_bytes.decode("utf-8", errors="replace")
        lines = buffer.split("\n")
        buffer = lines.pop()  # keep incomplete last line

        for line in lines:
            line = line.strip()
            if not line.startswith("data:"):
                continue
            json_str = line[len("data:") :].strip()
            if not json_str or json_str == "[DONE]":
                continue
            try:
                event = json.loads(json_str)
                audio_hex = event.get("data", {}).get("audio", "")
                status = event.get("data", {}).get("status")
                # status=1 is an audio chunk; status=2 is the aggregated final
                # audio (skip it when streaming to avoid duplicate data)
                if audio_hex and status != 2:
                    audio_chunks.append(bytes.fromhex(audio_hex))
            except (json.JSONDecodeError, ValueError):
                pass  # skip malformed lines

    if not audio_chunks:
        raise RuntimeError("No audio data received from MiniMax TTS API")

    with open(output_path, "wb") as f:
        for chunk in audio_chunks:
            f.write(chunk)

    print(f"Audio saved to: {output_path} ({sum(len(c) for c in audio_chunks):,} bytes)")
    return output_path


# Test TTS with the Claude response
audio_file = text_to_speech(test_response, output_path="claude_response.mp3")
print("Text-to-speech conversion complete!")

## Step 5: Play the Audio (Optional)

If you're running this notebook locally with audio output available, you can play the generated audio directly in the notebook.

In [ ]:
# Play audio inline (works in Jupyter environments with audio support)
from IPython.display import Audio, display

display(Audio(audio_file, autoplay=False))
print(f"Playing: {test_response}")

## Step 6: Build a Multi-Turn Voice Conversation

Let's put it all together to create a multi-turn conversation pipeline where Claude generates responses and MiniMax TTS converts them to speech.

In [ ]:
import os


class VoiceAssistant:
    """A voice assistant powered by Claude (LLM) and MiniMax (TTS)."""

    def __init__(
        self,
        voice_id: str = "English_Graceful_Lady",
        tts_model: str = "speech-2.8-hd",
        output_dir: str = "audio_outputs",
    ):
        self.voice_id = voice_id
        self.tts_model = tts_model
        self.output_dir = output_dir
        self.conversation_history: list = []
        self.turn_count = 0

        os.makedirs(output_dir, exist_ok=True)

    def chat(self, user_message: str) -> tuple[str, str]:
        """
        Process a user message: generate a Claude response and synthesize it.

        Returns:
            Tuple of (text_response, audio_file_path).
        """
        print(f"User: {user_message}")

        # Get Claude's text response
        text_response = get_claude_response(user_message, self.conversation_history)
        print(f"Claude: {text_response}")

        # Update conversation history
        self.conversation_history.append({"role": "user", "content": user_message})
        self.conversation_history.append({"role": "assistant", "content": text_response})

        # Convert to speech
        self.turn_count += 1
        audio_path = os.path.join(self.output_dir, f"turn_{self.turn_count:02d}.mp3")
        text_to_speech(
            text_response,
            voice_id=self.voice_id,
            model=self.tts_model,
            output_path=audio_path,
        )

        return text_response, audio_path

    def reset(self):
        """Reset conversation history."""
        self.conversation_history = []
        self.turn_count = 0
        print("Conversation history cleared.")


# Create the voice assistant
assistant = VoiceAssistant(voice_id="English_Graceful_Lady")

# Simulate a multi-turn conversation
conversation = [
    "Hello! Tell me something interesting about space exploration.",
    "That's fascinating! What's the next big milestone for Mars missions?",
    "Thank you, that's very informative!",
]

audio_files = []
for message in conversation:
    print("-" * 60)
    response_text, audio_path = assistant.chat(message)
    audio_files.append(audio_path)
    print()

print(f"\nGenerated {len(audio_files)} audio files in '{assistant.output_dir}/'")

## Step 7: Play All Conversation Audio

Listen to the generated conversation audio files.

In [ ]:
from IPython.display import Audio, display

for i, audio_path in enumerate(audio_files, 1):
    print(f"\nTurn {i}:")
    display(Audio(audio_path, autoplay=False))

## Step 8: Try Different Voices

MiniMax offers several English voices. Let's hear the same text in different voices.

In [ ]:
from IPython.display import Audio, display

sample_text = (
    "Welcome to the future of voice AI. MiniMax provides natural, expressive speech "
    "synthesis that brings your applications to life."
)

voices = [
    ("English_Graceful_Lady", "Graceful Lady (Female)"),
    ("English_Insightful_Speaker", "Insightful Speaker (Male)"),
    ("English_radiant_girl", "Radiant Girl (Female)"),
    ("English_Persuasive_Man", "Persuasive Man (Male)"),
]

os.makedirs("voice_demos", exist_ok=True)

for voice_id, voice_name in voices:
    print(f"\nVoice: {voice_name}")
    audio_path = f"voice_demos/{voice_id}.mp3"
    text_to_speech(sample_text, voice_id=voice_id, output_path=audio_path)
    display(Audio(audio_path, autoplay=False))

## Summary

In this cookbook, you learned how to:

1. **Generate conversational text** with Claude, optimized for speech output
2. **Synthesize speech** using MiniMax's TTS API (`speech-2.8-hd` model)
3. **Parse SSE streaming responses** from MiniMax to collect hex-encoded audio chunks
4. **Build a multi-turn voice assistant** that maintains conversation context
5. **Explore different voices** available in the MiniMax TTS system

## Next Steps

- Add speech-to-text (STT) to create a fully hands-free voice assistant
- Use `speech-2.8-turbo` for lower latency at the cost of some quality
- Experiment with `speed`, `vol`, and `pitch` parameters in `voice_setting`
- Try multilingual voices for non-English use cases
- Integrate with a web app or mobile application for real-world deployment

## Resources

- [MiniMax Platform](https://www.minimax.io/)
- [MiniMax TTS API Reference](https://platform.minimax.io/docs/api-reference/speech-t2a-http)
- [MiniMax Voice ID List](https://platform.minimax.io/faq/system-voice-id)
- [Anthropic API Documentation](https://docs.anthropic.com/)
- [Claude Models Overview](https://docs.anthropic.com/en/docs/about-claude/models/overview)